In [43]:
import pandas as pd
import json

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

In [ ]:
df = pd.read_csv("data/ru_62_gpt_preds_subsentences_adverbs.csv")

rows = list()
for idx, row in df.iterrows():
    preds = json.loads(row["preds"])
    for token in preds:
        temp = dict()
        temp["sent"] = row["sents"]
        for k in token.keys():
            temp[k] = token[k]
        rows.append(temp)

preds_df = pd.DataFrame(rows)

preds_df

In [ ]:
columns = ["sent", "word", "manual_fee", "manual_frame"]
manual_df = pd.read_csv("data/russian_fees.csv", header=0, names=columns)

manual_df

In [ ]:
combined_df = pd.concat([preds_df, manual_df[["manual_fee", "manual_frame"]]], axis=1)

combined_df

Total non-matches, including fee=False:

In [ ]:
combined_df[combined_df["frame"]!=combined_df["manual_frame"]]

In [48]:
print(f"""Complete match at {combined_df[combined_df["frame"]==combined_df["manual_frame"]].shape[0] / combined_df.shape[0] * 100}%""")

Complete match at 66.28126474752241%


Total non-matches, excluding fee=False:

In [ ]:
combined_df[(combined_df["frame"]!=combined_df["manual_frame"])&(combined_df["fee"]!=False)&(combined_df["manual_fee"]!=False)]

In [50]:
print(f"""Only fee=True complete match at {combined_df[(combined_df["frame"]==combined_df["manual_frame"])&(combined_df["fee"]!=False)&(combined_df["manual_fee"]!=False)].shape[0] / combined_df[(combined_df["fee"]!=False)&(combined_df["manual_fee"]!=False)].shape[0] * 100}%""")

Only fee=True complete match at 57.038391224862885%


Percentage of To_Review FEEs:

In [51]:
print(f"""For automatic annotation: {combined_df[combined_df["frame"]=="To_Review"].shape[0] / combined_df[combined_df["fee"]==True].shape[0] * 100}%""")

For automatic annotation: 0.4731182795698925%


In [52]:
print(f"""For manual annotation: {combined_df[combined_df["manual_frame"]=="To_Review"].shape[0] / combined_df[combined_df["manual_fee"]==True].shape[0] * 100}%""")

For manual annotation: 1.8919984233346472%


Percentage of manual FEEs:

In [53]:
print(f"""For manual annotation: {combined_df[combined_df["manual_fee"]==True].shape[0] / combined_df.shape[0] * 100}%""")

For manual annotation: 59.86314299197735%


In [54]:
print(f"""Accuracy: {accuracy_score(combined_df["manual_frame"], combined_df["frame"])}""")

Accuracy: 0.6628126474752242


In [55]:
print(f"""Micro metrics:\n
Precision: {precision_score(combined_df["manual_frame"], combined_df["frame"], average="micro")}
Recall: {recall_score(combined_df["manual_frame"], combined_df["frame"], average="micro")}
F1-score: {f1_score(combined_df["manual_frame"], combined_df["frame"], average="micro")}
""")

Micro metrics:

Precision: 0.6628126474752242
Recall: 0.6628126474752242
F1-score: 0.6628126474752242



In [56]:
print(f"""Macro metrics:\n
Precision: {precision_score(combined_df["manual_frame"], combined_df["frame"], average="macro", zero_division=0)}
Recall: {recall_score(combined_df["manual_frame"], combined_df["frame"], average="macro", zero_division=0)}
F1-score: {f1_score(combined_df["manual_frame"], combined_df["frame"], average="macro", zero_division=0)}
""")

Macro metrics:

Precision: 0.3605216597304347
Recall: 0.3516913762784996
F1-score: 0.3260300458730653



In [57]:
print(f"""Weighted metrics:\n
Precision: {precision_score(combined_df["manual_frame"], combined_df["frame"], average="weighted", zero_division=0)}
Recall: {recall_score(combined_df["manual_frame"], combined_df["frame"], average="weighted", zero_division=0)}
F1-score: {f1_score(combined_df["manual_frame"], combined_df["frame"], average="weighted", zero_division=0)}
""")

Weighted metrics:

Precision: 0.7099918702591687
Recall: 0.6628126474752242
F1-score: 0.6511996773567504



In [58]:
combined_df[["manual_frame"]].value_counts()

manual_frame         
-                        1698
Locative_relation          96
Calendric_unit             81
Negation                   80
Cardinal_numbers           77
                         ... 
Craft                       1
Process_continue            1
Court_examination           1
Contrition                  1
Intentional_deception       1
Name: count, Length: 488, dtype: int64